# Patterns in Text: Regular Expressions

In this lesson we learn what a [regular expression](https://dvirross.com/courses/intro-data-science/glossary#regular-expression) is, how the Python module `re` finds the pieces of a text that fit a pattern, how characters, repetitions and groups describe the pattern, and when a pattern is the right tool and when Beautiful Soup is.

## Where is the number of pages?

Authors who post a paper on [arXiv](https://arxiv.org) can add a line of comments, written as free text: `7 pages, 5 figures. Accepted to ASP-DAC 2027`. We want the number of pages of every paper that says it, for example to see how long papers in Hardware Architecture are. Beautiful Soup brings us to the element that holds the comment, as in [From One Element to All Entries: find_all, Numbers and Missing Values](https://dvirross.com/courses/intro-data-science/data-collection-find-all). The number inside the text has no element of its own: it is a piece of text with a shape, digits, a space and the word `pages`.

Text can be cut with `split` and `replace`, and that works while the number sits in a known place. A **regular expression** describes the shape instead of the place: the program asks for digits followed by the word, wherever they are in the text. Anyone who collects text from the web meets this need: prices, dates, phone numbers, identifiers and sizes sit inside sentences.

## Setup: the libraries and the check function

No account and no key are needed. The next cell imports `requests` and `beautifulsoup4` (Colab has both; the cell installs them if the session does not). The pages that the code reads are saved copies of arXiv pages, kept on the public site of the course, so the numbers are the same for everyone. This notebook sends no request to arXiv itself.

In [1]:
try:
    import bs4, requests
except ImportError:
    !pip install -q requests beautifulsoup4
    import bs4, requests
print('requests', requests.__version__, '| beautifulsoup4', bs4.__version__)

requests 2.34.2 | beautifulsoup4 4.15.0


The next cell defines the function `check(task_number)`. After you run the cell of a task, run the cell under it: `check` runs your cell again, collects what it prints, and compares it with the expected output. It tells you whether the output is right, and where it differs if not; it never shows the solution.

In [2]:
import ast, contextlib, hashlib, io

# The expected output of each task, as hashes of its lines (the solutions are not stored here).
EXPECTED = {int(k): v for k, v in {
    '1': [
        '79eb0b37c90d'
    ],
    '2': [
        '4fc82b26aecb'
    ],
    '3': [
        '9b7d87bc5316'
    ]
}.items()}


def check(task):
    """Run the last cell that ran (not a check) again and compare what it prints with the expected output of the task."""
    if task not in EXPECTED:
        print('There is no such task: ', task)
        return
    cells = [s for s in In[:-1] if s.strip() and not s.lstrip().startswith('check(')]   # In: the cells that ran, kept by IPython
    tree = ast.parse(cells[-1] if cells else '')
    last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None   # a bare last expression counts as printed
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            exec(compile(tree, '<task>', 'exec'), globals())
            if last is not None:
                value = eval(compile(ast.Expression(last.value), '<task>', 'eval'), globals())
                if value is not None:
                    print(repr(value))
    except Exception as error:
        print('The cell raised an error:', type(error).__name__ + ':', error)
        return
    got = [hashlib.sha256(line.rstrip().encode()).hexdigest()[:12] for line in out.getvalue().rstrip().splitlines()]
    want = EXPECTED[task]
    if not got:
        print('Nothing was printed. Print the answer in the task cell, run it, and run `check` straight after it.')
    elif len(got) != len(want):
        print('The cell printed {got} line(s) and the task expects {want}. Re-read the task and the hint.'.format(got=len(got), want=len(want)))
    elif got != want:
        print('Line {k} is different from the expected output. Re-read the task and the hint.'.format(k=next(i for i in range(len(want)) if got[i] != want[i]) + 1))
    else:
        print('OK: the output is as expected.')


The first box requests the saved copy of the list of October 2026 and builds the soup, as in [From a Web Page to Data: Beautiful Soup](https://dvirross.com/courses/intro-data-science/data-collection-beautifulsoup), so that this page stands on its own. It then puts the text of the comment of every paper in the list `comments`. Of the 25 papers, 12 have a line of comments.

In [3]:
import requests
from bs4 import BeautifulSoup

base_url = (
    'https://dvirross.com/materials/courses/'
    'intro-data-science/'
)
list_url = base_url + 'arxiv-cs-ar-2026-10.htm'
page_text = requests.get(list_url).text
soup = BeautifulSoup(page_text, 'html.parser')

boxes = soup.find_all('div', class_='list-comments')
comments = [
    box.text.replace('Comments:', '').strip()
    for box in boxes
]
print(len(comments))
print(comments[1])

12
7 pages, 5 figures. Accepted to ASP-DAC 2027


## `re.search`: the first piece that fits

The module `re` comes with Python, and `import re` makes it available. A **[pattern](https://dvirross.com/courses/intro-data-science/glossary#pattern)** is a string that describes the text we look for. It is written as a **[raw string](https://dvirross.com/courses/intro-data-science/glossary#raw-string)**: a string with the letter `r` before the opening quote, such as `r'pages'`. In a raw string Python leaves every backslash as it is, and the patterns below use many backslashes, so every pattern in this lesson is a raw string.

The function `re.search(pattern, text)` looks for the first place in the text where the pattern fits. It returns a **[match object](https://dvirross.com/courses/intro-data-science/glossary#match)** that describes the place, or `None` when the pattern fits nowhere. The method `group()` of a match object returns the piece of text that fit.

In [4]:
import re

text = comments[1]
match = re.search(r'pages', text)
print(match)
print(match.group())
print(re.search(r'tables', text))

<re.Match object; span=(2, 7), match='pages'>
pages
None


The match object says that the piece starts at position 2 and ends before position 7. As with `find`, a result that can be `None` is tested with `is None` before `group()` is called on it.

## Characters and classes

A pattern is read from left to right, and each piece of it fits one character of the text. A letter fits itself. A **class** is a piece that fits any one character of a kind:

| Piece | Fits one character that is | Example in a comment |
|---|---|---|
| `\d` | a digit, 0 to 9 | `7` |
| `[0-9]` | a digit: any character in the range in the brackets | `7` |
| `[pf]` | one of the characters in the brackets | the `p` of `pages` or the `f` of `figures` |
| `\w` | a letter, a digit or an underscore | `p`, `5` |
| `\s` | white space: a space, a tab or a line break | the space after a comma |
| `.` | any character except a line break | any character |

Some characters have a meaning of their own in a pattern, such as `.`, `[` and `(`. To fit the character itself, write a backslash before it: `\.` fits a dot and `\[` fits a square bracket.

In [5]:
print(re.search(r'\d pages', comments[1]))
print(re.search(r'[0-9] pages', comments[1]))
print(re.search(r'figures.\s\w', comments[1]))

<re.Match object; span=(0, 7), match='7 pages'>
<re.Match object; span=(0, 7), match='7 pages'>
<re.Match object; span=(11, 21), match='figures. A'>


**Your turn.** The comment of another paper begins with `15 pages.` What does the pattern `\d pages` fit in it? Predict the piece, then run the box.

In [6]:
print(re.search(r'\d pages', comments[9]).group())

5 pages


**Solution**

The pattern fits `5 pages`, not `15 pages`. The piece `\d` fits one digit, and the search takes the first place where the whole pattern fits: the digit `1` is followed by `5`, not by a space, so the match starts at the `5`. A number of any length needs a piece that repeats.

## Quantifiers: how many times

A **quantifier** written after a piece says how many times the piece repeats. It belongs to the piece just before it.

| Quantifier | Means | Example |
|---|---|---|
| `+` | one or more times | `\d+` fits `7` and `15` |
| `*` | zero or more times | `\d*` fits `15`, and also fits nothing |
| `?` | zero or one time: the piece is optional | `pages?` fits `page` and `pages` |
| `{n}` | exactly n times | `\d{4}` fits `2026` |

The function `re.findall(pattern, text)` returns a list of all the pieces of the text that fit, from left to right, without overlapping. Each item is a string. In the second line of the box below the pattern finds every run of digits in a comment, and in the third line it finds the identifier of each paper in the list: four digits, a dot and five digits. The variable `soup.text` is all the text of the page, without its tags.

In [7]:
print(re.search(r'\d+ pages', comments[9]).group())
print(re.findall(r'\d+', comments[2]))
ids = re.findall(r'\d{4}\.\d{5}', soup.text)
print(len(ids), ids[:3])

15 pages
['4', '3', '3', '2026']
25 ['2610.00106', '2610.00207', '2610.00311']


## Groups: taking a part of the match

The pattern `\d+ pages` fits the text `15 pages`, but the number we want is the first part. Round brackets in a pattern make a **[capturing group](https://dvirross.com/courses/intro-data-science/glossary#capturing-group)**: the pattern fits as if the brackets were not there, and the piece that fits inside them is also saved. The method `group(1)` returns the piece of the first pair of brackets, `group(2)` the piece of the second, counted by the opening bracket from left to right. A group returns text, so `int` converts it to a number.

In [8]:
match = re.search(r'(\d+) pages, (\d+) figures', comments[1])
print(match.group())
print(match.group(1), match.group(2))
print(int(match.group(1)) + 1)

7 pages, 5 figures
7 5
8


When the pattern has groups, `re.findall` returns the pieces of the groups instead of the whole piece: a list of strings for one group, and a list of tuples for several groups.

In [9]:
print(re.findall(r'(\d+) pages', comments[2]))
print(re.findall(r'(\d+) pages, (\d+) figures', comments[2]))

['4']
[('4', '3')]


### Practice

**Your turn.** The list `comments` holds the comment of every paper that has one. Print the list of the numbers of pages, as whole numbers, of the papers whose comment says how many pages, in the order of `comments`.

> **Hint**
>
> Go over the comments in a loop. For each one, `re.search` with a group around the digits returns a match object or `None`; test it before you call `group(1)`, and convert the piece with `int`.

**Solution**

In [10]:
pages = []
for text in comments:
    match = re.search(r'(\d+) pages', text)
    if match is not None:
        pages.append(int(match.group(1)))
print(pages)

[7, 4, 4, 8, 10, 15]


**Expected output**

```text
[7, 4, 4, 8, 10, 15]
```

In [11]:
check(1)

OK: the output is as expected.


**Common mistake**

```python
pages = []
for text in comments:
    match = re.search(r'(\d) pages', text)
    if match is not None:
        pages.append(int(match.group(1)))
print(pages)
```

The piece `\d` fits one digit, so the pages of `10 pages` and `15 pages` become 0 and 5. The result, `[7, 4, 4, 8, 0, 5]`, has the right length and the wrong numbers.

## The size and the date of a submission

The page of a paper has a section called the submission history. Its text holds, in one line, the date and the size of the first version, `[v1]`, of the paper: `[v1] Wed, 9 Sep 2026 02:40:29 UTC (3,330 KB)`. A number inside text can be cut with `split`, `replace` and `strip`, as in [Following Links to Many Pages](https://dvirross.com/courses/intro-data-science/data-collection-linked-pages); a pattern describes the number and its unit directly.

The function `history_of` requests the saved page of a paper by its identifier and returns the text of its submission history. Saved pages exist for the first eight papers of the list.

In [12]:
def history_of(paper_id):
    address = base_url + 'arxiv-abs-' + paper_id + '.htm'
    page = BeautifulSoup(requests.get(address).text, 'html.parser')
    return page.find('div', class_='submission-history').text

history = history_of(ids[0])
line = re.search(r'\[v1\]\s+(.*)', history).group(1)
print(line)

Wed, 9 Sep 2026 02:40:29 UTC (3,330 KB)


The pattern `\[v1\]\s+(.*)` fits the square bracket, `v1` and a bracket, then `\s+`, the line break and the spaces that follow it in the page, and the group `(.*)`: any number of characters up to the end of the line, because the dot does not fit a line break.

In the line, the size is digits and commas followed by `KB`, and the date is a day, a month and a year. The class `[0-9,]` fits a digit or a comma, a comma has no special meaning inside the brackets, so `[0-9,]+` fits `3,330`. The piece `\d\d?` is a digit and an optional second digit: a day of the month has one or two digits. The text `3,330` is still text, and `replace` removes the comma before `int` converts it.

In [13]:
size = re.search(r'([0-9,]+) KB', line).group(1)
print(size)
print(int(size.replace(',', '')))
print(re.findall(r'(\d\d?) (\w+) (\d{4})', line))

3,330
3330
[('9', 'Sep', '2026')]


## The dot and the shortest match

The subjects of a paper are written in a line such as `Hardware Architecture (cs.AR); Artificial Intelligence (cs.AI)`. The code of each subject is inside round brackets. In the pattern, `\(` and `\)` fit the round brackets themselves, and the pair of brackets without a backslash is the group that saves what is between them.

The box below defines the function `subjects_of`, which returns the subjects line of one entry, and builds the list `subjects` for the 25 papers. The second paper has three subjects.

In [14]:
def subjects_of(entry):
    box = entry.find('div', class_='list-subjects')
    return box.text.replace('Subjects:', '').strip()

subjects = [subjects_of(e) for e in soup.find_all('dd')]
print(subjects[1])

Hardware Architecture (cs.AR); Artificial Intelligence (cs.AI); Image and Video Processing (eess.IV)


The piece `.*` fits any number of characters, and it takes as many as it can while the whole pattern still fits: it is **greedy**. In `\((.*)\)` the group starts after the first opening bracket and runs to the last closing bracket of the line. A `?` after a quantifier makes it take as few characters as it can, the **shortest match**: `.*?` stops at the first closing bracket that lets the pattern fit.

In [15]:
print(re.findall(r'\((.*)\)', subjects[1]))
print(re.findall(r'\((.*?)\)', subjects[1]))

['cs.AR); Artificial Intelligence (cs.AI); Image and Video Processing (eess.IV']
['cs.AR', 'cs.AI', 'eess.IV']


The same character, `?`, means two things: after a piece it makes the piece optional, and after a quantifier it makes the quantifier take as little as possible.

### Practice

**Your turn.** The list `subjects` holds the subjects line of each of the 25 papers. The code of a subject is inside round brackets. Print the number of papers that have more than one subject.

> **Hint**
>
> `re.findall` with a group around the code returns the codes of one paper as a list; a paper has more than one subject when the list has more than one item. Make sure the group stops at the first closing bracket.

**Solution**

In [16]:
many = 0
for text in subjects:
    codes = re.findall(r'\((.*?)\)', text)
    if len(codes) > 1:
        many = many + 1
print(many)

11


**Expected output**

```text
11
```

In [17]:
check(2)

OK: the output is as expected.


**Common mistake**

```python
many = 0
for text in subjects:
    codes = re.findall(r'\((.*)\)', text)
    if len(codes) > 1:
        many = many + 1
print(many)
```

The greedy `.*` takes everything from the first opening bracket to the last closing bracket, so every paper gives exactly one piece, and the result is 0.

## Testing a pattern in a tool

A pattern that looks right can fit something else, so we test it on real text before we run it on a thousand pages. The free site [regex101](https://regex101.com) is a regular expression tester: the text goes in one box and the pattern in another, every piece that fits is highlighted, and a panel explains each part of the pattern. Choose the Python flavor on the left of the page, because the other flavors differ in details. In the tool a pattern is written without the quotes and without the letter `r`.

To try it, paste the comment `7 pages, 5 figures. Accepted to ASP-DAC 2027` as the text and `(\d+) pages, (\d+) figures` as the pattern: the tool shows the whole match and the two groups. Then change the text so that it says `1 page` and see why the pattern no longer fits.

## When a regular expression is not the tool

A regular expression reads text and knows nothing of a page. Beautiful Soup builds the tree of the elements and finds one by its tag, its class or its attribute. A pattern applied to the raw HTML text of a page depends on how the tags happen to be written, and it fails quietly: it returns an empty list and no error. The box counts, with the pattern and with Beautiful Soup, the identifiers and the comment elements of the list:

In [18]:
print(len(re.findall(r'\d{4}\.\d{5}', page_text)))
print(len(re.findall(r'\d{4}\.\d{5}', soup.text)))
print(len(re.findall(r'<div class="list-comments">', page_text)))
print(len(soup.find_all('div', class_='list-comments')))

300
25
0
12


The HTML repeats each identifier 12 times, in the addresses and the attributes of its links (25 times 12 is 300), and the text of the page gives 25. The class of the element is `list-comments mathjax`, with a second word, so the pattern finds nothing, and `class_='list-comments'` finds the 12 elements.

The division of work is therefore this: Beautiful Soup finds the element that holds the text, a pattern takes the number, the date or the code out of the text of that element. A pattern also describes a shape and not a meaning: in the abstract of the paper EdgeDAE, `\d+\.\d+` fits the percentages `52.5` and `38.7`, and also `2.10`, which is a ratio of throughput. We look at the pieces that fit, count them, and test the pattern on texts that should not fit.

## Keep in mind

- Beautiful Soup finds an element by the structure of the page; a regular expression finds a pattern inside a piece of text. Take the `text` of the element, then search it with a pattern.
- A pattern is a raw string, `r'...'`. `re.search` returns a match object, whose `group()` is the piece that fit, or `None`; `re.findall` returns a list of all the pieces that fit, as text.
- `\d`, `\w`, `\s`, `[0-9]`, `[abc]` and `.` fit one character; a backslash before `.`, `[` or `(` fits the character itself.
- `+`, `*`, `?` and `{n}` say how many times a piece repeats; round brackets save a part of the match, and with groups `re.findall` returns the groups.
- `.*` takes as much as it can and `.*?` as little as it can; a group of the form `\((.*?)\)` stops at the first closing bracket.
- A piece taken out of text is text: `int` converts digits, after `replace` removes a comma. A pattern that fits the examples can still fit other text, so test it and count what it finds.

> **Self-study, importance 1.** This part is for self-study: it is not taught in the lecture, and you read it on your own. Compiling a pattern once and using it many times. It helps with: naming a pattern and reusing it in a function.

## `re.compile`: a pattern with a name

A pattern that a program uses many times can be built once. The function `re.compile(pattern)` returns a compiled pattern, an object that holds the pattern and has the methods `search` and `findall` of its own, so that `compiled.search(text)` does what `re.search(pattern, text)` does. The function `pages_of` below returns the number of pages of a comment, or `None`, with a pattern that is built once.

In [19]:
pages_pattern = re.compile(r'(\d+) pages')

def pages_of(text):
    match = pages_pattern.search(text)
    if match is None:
        return None
    return int(match.group(1))

all_pages = [pages_of(text) for text in comments]
print(all_pages)
known = [pages for pages in all_pages if pages is not None]
print(len(known), max(known), sum(known) / len(known))

[None, 7, 4, 4, None, None, 8, None, 10, 15, None, None]
6 15 8.0


The six papers that say how many pages they have are the same six as in the function that cuts the text with `split`, in [From One Element to All Entries: find_all, Numbers and Missing Values](https://dvirross.com/courses/intro-data-science/data-collection-find-all); here the pattern finds the number wherever it stands in the comment.

> **Self-study, importance 2.** This part is for self-study: it is not taught in the lecture, and you read it on your own. A practice task that reads several pages and takes a size from each. It helps with: a regular expression applied to many pages.

### Practice: several pages

**Your turn.** The function `history_of` returns the text of the submission history of a paper by its identifier, and the list `ids` holds the identifiers of the 25 papers; saved pages exist for the first eight. Print the largest size, as a whole number of KB, among the first eight papers.

> **Hint**
>
> Go over `ids[:8]`, and for each identifier take the text of the size with a group around the digits and the comma, remove the comma, and convert it with `int`. Compare numbers, not texts.

**Solution**

In [20]:
sizes = []
for paper_id in ids[:8]:
    history = history_of(paper_id)
    size = re.search(r'([0-9,]+) KB', history).group(1)
    sizes.append(int(size.replace(',', '')))
print(max(sizes))

19279


**Expected output**

```text
19279
```

In [21]:
check(3)

OK: the output is as expected.


**Common mistake**

```python
sizes = []
for paper_id in ids[:8]:
    history = history_of(paper_id)
    size = re.search(r'([0-9,]+) KB', history).group(1)
    sizes.append(size)
print(max(sizes))
```

The sizes stay as text, and `max` compares texts character by character: `'866'` is larger than `'19,279'` because `'8'` comes after `'1'`. The program prints 866, a size that is not the largest.